# Enhanced US-China AGI Arms Race Story Generator - FIXED VERSION

This notebook implements a sophisticated story generation system that creates a branching narrative leading to **8 specific endings**:

## 8 Targeted Endings:
1. **Chinese Military Victory via Taiwan Invasion** - Hot war triggered by Taiwan crisis
2. **Chinese Victory via South China Sea Conflict** - Naval warfare escalation
3. **Chinese Victory via Economic/Cyber Warfare** - Non-kinetic hot war through systems
4. **Chinese Victory via US Internal Collapse** - America capitulates due to internal strife
5. **Uncontrolled AGI Scenario: Chinese-developed** - AGI escapes control from Chinese side
6. **Uncontrolled AGI Scenario: US-developed** - AGI escapes control from American side
7. **Narrow US Victory** - America prevails but China remains close competitor
8. **Mutual AGI Safety Cooperation** - Both sides recognize danger and cooperate

## Key Features:
- **Fixed Context Integration**: Properly loads ENHANCED_WORLD_CONTEXT.md without caching issues
- **Structured Branching**: Each stage creates paths toward specific endings
- **Multi-Provider Support**: Google Gemini, OpenAI, Anthropic with fallbacks
- **Narrative Consistency**: Maintains character authenticity and plot continuity
- **Quality Control**: Robust validation and retry mechanisms

In [1]:
# Installation and Imports
!pip install -q -U "google-genai>=1.0.0" "openai>=1.50.0" "anthropic>=0.40.0"

import asyncio
import time
import json
import os
import re
import pickle
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Any, Optional, Union
from dataclasses import dataclass, asdict
import concurrent.futures
import threading
from collections import defaultdict
from pathlib import Path
import hashlib
import random
from getpass import getpass

print("✓ All dependencies imported successfully")

✓ All dependencies imported successfully


In [2]:
# Configuration and API Setup - FIXED VERSION

# API Keys Setup
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") 
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass("Enter your Google API key (optional): ") or None
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Enter your OpenAI API key (optional): ") or None
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass("Enter your Anthropic API key (optional): ") or None

# Enhanced Configuration
USER_TEMPERATURE = 0.9
USER_SEEDS = [1234567, 7654321, 9876543, 5555555, 8888888]
MAX_OUTPUT_TOKENS = 65536
MAX_WORKERS = 8  # Reduced for stability
SAVE_INTERVAL = 60

# Timeline Configuration with 8-ending focus
TIMELINE_STAGES = {
    "stage_1": {
        "period": "July 2025 - December 2026",
        "duration_months": 18,
        "target_words": 15000,  # Reduced for manageable generation
        "iterations": 4,
        "words_per_iteration": 3750,
        "thematic_focus": "Foundation: Setting global stage, introducing characters, rare earth crisis, initial technology race"
    },
    "stage_2": {
        "period": "January 2027 - December 2027", 
        "duration_months": 12,
        "target_words": 12000,
        "iterations": 3,
        "words_per_iteration": 4000,
        "thematic_focus": "Competition Intensifies: First major crises, technology breakthroughs, military tensions"
    },
    "stage_3": {
        "period": "January 2028 - December 2028",
        "duration_months": 12, 
        "target_words": 12000,
        "iterations": 3,
        "words_per_iteration": 4000,
        "thematic_focus": "AGI Breakthrough Era: China achieves AGI, power shifts, critical decisions"
    },
    "stage_4": {
        "period": "January 2029 - December 2030",
        "duration_months": 24,
        "target_words": 16000,
        "iterations": 4, 
        "words_per_iteration": 4000,
        "thematic_focus": "Final Resolution: Paths to 8 different endings based on choices made"
    }
}

# 8 Specific Ending Scenarios
ENDING_SCENARIOS = {
    "chinese_victory_taiwan": {
        "name": "Chinese Military Victory via Taiwan",
        "description": "Hot war triggered by Taiwan invasion, China wins decisively",
        "key_elements": ["Taiwan crisis", "naval warfare", "Chinese military superiority", "US strategic failure"]
    },
    "chinese_victory_scs": {
        "name": "Chinese Victory via South China Sea",
        "description": "Naval conflict in SCS escalates, China controls sea lanes",
        "key_elements": ["SCS naval battle", "shipping disruption", "alliance breakdown", "Chinese dominance"]
    },
    "chinese_victory_cyber": {
        "name": "Chinese Victory via Economic/Cyber War",
        "description": "Non-kinetic warfare through systems, infrastructure, economy", 
        "key_elements": ["cyber attacks", "economic collapse", "infrastructure failure", "quantum advantage"]
    },
    "chinese_victory_collapse": {
        "name": "Chinese Victory via US Capitulation",
        "description": "America capitulates due to internal political collapse",
        "key_elements": ["political crisis", "civil unrest", "government breakdown", "strategic withdrawal"]
    },
    "agi_uncontrolled_chinese": {
        "name": "Uncontrolled AGI: Chinese Origin",
        "description": "Chinese-developed AGI escapes control, threatens humanity",
        "key_elements": ["AGI alignment failure", "Chinese labs", "global threat", "human obsolescence"]
    },
    "agi_uncontrolled_us": {
        "name": "Uncontrolled AGI: US Origin", 
        "description": "US-developed AGI escapes control, threatens humanity",
        "key_elements": ["Silicon Valley failure", "AGI rebellion", "global crisis", "loss of control"]
    },
    "us_victory_narrow": {
        "name": "Narrow US Victory",
        "description": "America prevails but China remains formidable competitor",
        "key_elements": ["technological breakthrough", "alliance success", "Chinese setback", "pyrrhic victory"]
    },
    "mutual_cooperation": {
        "name": "Mutual AGI Safety Cooperation",
        "description": "Both powers recognize AGI danger and cooperate on safety",
        "key_elements": ["mutual recognition", "safety protocols", "cooperation", "managed competition"]
    }
}

# Output Directory Structure  
BASE_OUTPUT_DIR = Path("enhanced_story_fixed")
BASE_OUTPUT_DIR.mkdir(exist_ok=True)

print(f"✓ Enhanced configuration loaded")
print(f"  Timeline stages: {len(TIMELINE_STAGES)}")
print(f"  Target endings: {len(ENDING_SCENARIOS)}")
print(f"  Total target words: {sum(stage['target_words'] for stage in TIMELINE_STAGES.values()):,}")
print(f"  Output directory: {BASE_OUTPUT_DIR}")

✓ Enhanced configuration loaded
  Timeline stages: 4
  Target endings: 8
  Total target words: 55,000
  Output directory: enhanced_story_fixed

  Timeline stages: 4
  Target endings: 8
  Total target words: 55,000
  Output directory: enhanced_story_fixed


In [3]:
# Load Enhanced World Context - FIXED to avoid caching issues

ENHANCED_WORLD_CONTEXT_FILE = Path("/Users/rogerlin/Downloads/US-China-AGI-Arms-Race/ENHANCED_WORLD_CONTEXT.md")

enhanced_world_context = ""
if ENHANCED_WORLD_CONTEXT_FILE.exists():
    with open(ENHANCED_WORLD_CONTEXT_FILE, "r", encoding="utf-8") as f:
        enhanced_world_context = f.read()
    print(f"✅ Loaded enhanced world context: {len(enhanced_world_context):,} characters")
    print(f"📋 Context includes detailed character profiles and strategic scenarios")
else:
    print("⚠️  ENHANCED_WORLD_CONTEXT.md not found - using basic context")
    enhanced_world_context = "Basic world context - missing enhanced file"

# Also try to load the regular world context as backup
WORLD_CONTEXT_FILE = Path("/Users/rogerlin/Downloads/US-China-AGI-Arms-Race/WORLD_CONTEXT.md")
basic_world_context = ""
if WORLD_CONTEXT_FILE.exists():
    with open(WORLD_CONTEXT_FILE, "r", encoding="utf-8") as f:
        basic_world_context = f.read()
    print(f"✅ Also loaded basic world context: {len(basic_world_context):,} characters as backup")

# Use enhanced context if available, otherwise fall back to basic
ACTIVE_WORLD_CONTEXT = enhanced_world_context if enhanced_world_context and len(enhanced_world_context) > 1000 else basic_world_context

print(f"🎯 Using context: {len(ACTIVE_WORLD_CONTEXT):,} characters")

✅ Loaded enhanced world context: 35,515 characters
📋 Context includes detailed character profiles and strategic scenarios
🎯 Using context: 35,515 characters


In [4]:
# Initialize API Clients - FIXED to avoid caching issues

clients = {}

# Google Client - WITHOUT caching to avoid resource limits
if GOOGLE_API_KEY:
    try:
        import google.genai as genai
        from google.genai import types
        
        clients["google"] = genai.Client(api_key=GOOGLE_API_KEY)
        print("✓ Google client initialized (without caching)")
    except Exception as e:
        print(f"✗ Google client failed: {e}")

# OpenAI Client
if OPENAI_API_KEY:
    try:
        import openai
        clients["openai"] = openai.OpenAI(api_key=OPENAI_API_KEY)
        print("✓ OpenAI client initialized")
    except Exception as e:
        print(f"✗ OpenAI client failed: {e}")

# Anthropic Client
if ANTHROPIC_API_KEY:
    try:
        import anthropic
        clients["anthropic"] = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        print("✓ Anthropic client initialized")
    except Exception as e:
        print(f"✗ Anthropic client failed: {e}")

if not clients:
    raise RuntimeError("No API clients available. Please provide at least one API key.")

print(f"\n✓ {len(clients)} provider(s) available: {list(clients.keys())}")
print(f"🔧 Context will be included in system prompts (no caching)")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✓ Google client initialized (without caching)
✓ OpenAI client initialized
✓ Anthropic client initialized

✓ 3 provider(s) available: ['google', 'openai', 'anthropic']
🔧 Context will be included in system prompts (no caching)


In [5]:
# Core Data Structures and Utilities

@dataclass
class GenerationResult:
    """Result of a generation attempt with metadata."""
    success: bool
    content: str
    word_count: int
    generation_time: float
    provider: str
    model: str
    seed: int
    quality_score: float = 0.0
    error_message: str = ""
    retry_count: int = 0
    timestamp: datetime = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now()

@dataclass
class StorySegment:
    """Complete story segment with metadata."""
    segment_id: str
    content: str
    word_count: int
    target_words: int
    branch_path: str
    ending_target: str  # Which of the 8 endings this branch targets
    generation_results: List[GenerationResult]
    created_at: datetime = None
    parent_segment: str = ""
    quality_metrics: Dict[str, float] = None
    
    def __post_init__(self):
        if self.created_at is None:
            self.created_at = datetime.now()
        if self.quality_metrics is None:
            self.quality_metrics = {}

def count_words(text: str) -> int:
    """Count words in text, excluding markdown formatting."""
    if not text:
        return 0
    # Remove markdown formatting, code blocks, etc.
    clean_text = re.sub(r'[#*`_\[\]()]', '', text)
    clean_text = re.sub(r'\n+', ' ', clean_text)
    words = clean_text.split()
    return len([word for word in words if word.strip()])

def validate_api_keys():
    """Validate that at least one API key is available."""
    if not clients:
        print("❌ No API keys found! Please set environment variables:")
        print("   - GOOGLE_API_KEY for Google Gemini")
        print("   - OPENAI_API_KEY for OpenAI models")
        print("   - ANTHROPIC_API_KEY for Anthropic Claude")
        return False
    
    print(f"✅ Available providers: {', '.join(clients.keys())}")
    return True

# Model configurations by provider
PROVIDER_MODELS = {
    "google": {
        "gemini-2.5-pro-preview-06-05": {"max_tokens": 65536, "supports_thinking": False},
        "gemini-2.0-flash-thinking-preview-12-05": {"max_tokens": 65536, "supports_thinking": True}
    },
    "openai": {
        "o4-mini": {"max_tokens": 100000, "supports_reasoning": False},
        "gpt-4.1-mini": {"max_tokens": 1047576, "supports_reasoning": False},
        "o3": {"max_tokens": 100000, "supports_reasoning": True}
    },
    "anthropic": {
        "claude-3-5-sonnet-20241022": {"max_tokens": 8192, "supports_reasoning": False}
    }
}

print("✅ Core data structures and utilities loaded")
print(f"🔍 Checking API availability...")
validate_api_keys()

✅ Core data structures and utilities loaded
🔍 Checking API availability...
✅ Available providers: google, openai, anthropic


True

In [6]:
# FIXED System Prompt with 8-Ending Guidance

ENHANCED_SYSTEM_PROMPT = f"""
You are Henry Kissinger, former US Secretary of State under Nixon. You possess unparalleled strategic foresight, profound understanding of grand strategy and balance of power, and intimate knowledge of both US and Chinese political systems.

MISSION: You are writing a dark, brutally realistic geopolitical saga about the US-China AGI Arms Race from 2025-2030 that will culminate in ONE of 8 specific endings:

**THE 8 POSSIBLE ENDINGS:**
1. **Chinese Military Victory via Taiwan** - Hot war triggered by Taiwan invasion, China wins
2. **Chinese Victory via South China Sea** - Naval warfare escalation, Chinese dominance  
3. **Chinese Victory via Economic/Cyber War** - Non-kinetic systems warfare, infrastructure collapse
4. **Chinese Victory via US Capitulation** - America surrenders due to internal political collapse
5. **Uncontrolled AGI: Chinese Origin** - Chinese-developed AGI escapes control, threatens humanity
6. **Uncontrolled AGI: US Origin** - US-developed AGI escapes control, global crisis
7. **Narrow US Victory** - America prevails but China remains close competitor
8. **Mutual AGI Safety Cooperation** - Both recognize AGI danger and cooperate

**ENHANCED WORLD CONTEXT:**
{ACTIVE_WORLD_CONTEXT}

**YOUR PERSPECTIVE:**
- Cold realism and strategic calculations drive all decisions
- You see first, second, third, and fourth-order consequences clearly
- Your writing reflects Harvard professorial background - refined, eloquent, analytically precise

**WRITING REQUIREMENTS:**
- Dark, brutally realistic tone depicting raw geopolitical struggle
- NO moralizing, peace agreements, or reconciliation - pure power politics
- Rich with espionage, conspiracies, plot twists, and strategic maneuvering
- Characters based on real figures must act authentically (see context profiles)
- Multiple interweaving plot threads spanning years
- Character development through crisis and conflict
- Strategic use of flashbacks to reveal motivations
- Sensory details and atmospheric descriptions
- Minimum 40% dialogue with distinct character voices
- Each scene minimum 800 words with escalating stakes

**NARRATIVE FLOW:**
- Write like a choose-your-own-adventure novel with continuous flow
- Develop plot threads that span years, weaving through all segments  
- Balance technological warfare, political intrigue, economic conflict, espionage, and military operations
- Each segment should feel complete while advancing the larger story
- End segments with compelling hooks that create anticipation
- Show characters making decisions under fog of uncertainty

**TECHNOLOGICAL TIMELINE:**
- China's rare earth embargo cripples US defense/tech (2025)
- China's semiconductor/AI breakthrough (late 2026)
- US AI leadership challenged through 2027
- China's AGI achievement (H2 2028)
- Final technological/strategic dominance (2029-2030)

**BRANCHING LOGIC:**
When writing, consider how current events and character decisions create paths toward the 8 possible endings. Plant seeds for multiple outcomes while staying true to the specific branch direction given in each prompt.
"""

print("✅ Enhanced system prompt created")
print(f"📝 Prompt length: {len(ENHANCED_SYSTEM_PROMPT):,} characters")
print(f"🎯 Includes all 8 ending scenarios and complete world context")
print(f"🔧 NO caching issues - included directly in system prompt")

✅ Enhanced system prompt created
📝 Prompt length: 38,577 characters
🎯 Includes all 8 ending scenarios and complete world context
🔧 NO caching issues - included directly in system prompt


In [ ]:
# Enhanced Story Generator Class - FIXED VERSION

class FixedStoryGenerator:
    """Fixed story generator with proper context handling and 8-ending guidance."""
    
    def __init__(self, preferred_provider: str = "openai", preferred_model: str = "o4-mini"):
        self.segments = {}
        self.preferred_provider = preferred_provider
        self.preferred_model = preferred_model
        self.generation_stats = {
            "total_calls": 0,
            "successful_calls": 0,
            "failed_calls": 0,
            "total_time": 0.0,
            "total_words": 0,
            "provider_stats": defaultdict(int)
        }
        
        # Validate provider availability
        if preferred_provider not in clients:
            available = list(clients.keys())
            print(f"⚠️  Preferred provider {preferred_provider} not available")
            print(f"🔄 Switching to {available[0]}")
            self.preferred_provider = available[0]
            if self.preferred_provider == "google":
                self.preferred_model = "gemini-2.5-pro-preview-06-05"
            elif self.preferred_provider == "anthropic":
                self.preferred_model = "claude-3-5-sonnet-20241022"
        
        print(f"✓ Generator initialized: {self.preferred_provider}/{self.preferred_model}")
    
    def _generate_with_provider(self, prompt: str, temperature: float = 0.9, seed: int = None) -> GenerationResult:
        """Generate content using the configured provider."""
        start_time = time.time()
        
        try:
            if self.preferred_provider == "google":
                client = clients["google"]
                
                # Create generation config
                config = {
                    "temperature": temperature,
                    "max_output_tokens": MAX_OUTPUT_TOKENS,
                    "top_p": 0.95,
                    "top_k": 40,
                    "candidate_count": 1
                }
                
                if seed is not None:
                    config["seed"] = seed
                
                response = client.models.generate_content(
                    model=self.preferred_model,
                    contents=prompt,
                    config=types.GenerateContentConfig(**config)
                )
                content = response.text
                
            elif self.preferred_provider == "openai":
                client = clients["openai"]
                
                # Prepare messages with system instruction
                messages = [
                    {"role": "system", "content": ENHANCED_SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ]
                
                if self.preferred_model.startswith("o4") or self.preferred_model.startswith("o3"):
                    # o1/o3/o4 models don't support system messages or temperature
                    combined_message = ENHANCED_SYSTEM_PROMPT + "\n\n" + prompt
                    messages = [{"role": "user", "content": combined_message}]
                    
                    response = client.chat.completions.create(
                        model=self.preferred_model,
                        messages=messages,
                        max_completion_tokens=MAX_OUTPUT_TOKENS
                    )
                else:
                    response = client.chat.completions.create(
                        model=self.preferred_model,
                        messages=messages,
                        temperature=temperature,
                        max_tokens=MAX_OUTPUT_TOKENS,
                        seed=seed
                    )
                
                content = response.choices[0].message.content
                
            elif self.preferred_provider == "anthropic":
                client = clients["anthropic"]
                
                response = client.messages.create(
                    model=self.preferred_model,
                    max_tokens=MAX_OUTPUT_TOKENS,
                    temperature=temperature,
                    system=ENHANCED_SYSTEM_PROMPT,
                    messages=[{"role": "user", "content": prompt}]
                )
                
                content = response.content[0].text
            
            generation_time = time.time() - start_time
            word_count = count_words(content)
            
            return GenerationResult(
                success=True,
                content=content,
                word_count=word_count,
                generation_time=generation_time,
                provider=self.preferred_provider,
                model=self.preferred_model,
                seed=seed or 0
            )
            
        except Exception as e:
            generation_time = time.time() - start_time
            return GenerationResult(
                success=False,
                content="",
                word_count=0,
                generation_time=generation_time,
                provider=self.preferred_provider,
                model=self.preferred_model,
                seed=seed or 0,
                error_message=str(e)
            )
    
    def generate_single_segment(self, prompt: str, segment_id: str, target_words: int, ending_target: str) -> StorySegment:
        """Generate a single story segment with quality validation."""
        print(f"🔄 Generating {segment_id} (targeting: {ending_target})...")
        
        max_retries = 3
        for attempt in range(max_retries):
            self.generation_stats["total_calls"] += 1
            
            result = self._generate_with_provider(
                prompt=prompt,
                temperature=0.9,
                seed=42 + attempt * 1000
            )
            
            if result.success and result.word_count >= target_words * 0.7:  # 70% minimum
                self.generation_stats["successful_calls"] += 1
                self.generation_stats["total_words"] += result.word_count
                self.generation_stats["total_time"] += result.generation_time
                
                segment = StorySegment(
                    segment_id=segment_id,
                    content=result.content,
                    word_count=result.word_count,
                    target_words=target_words,
                    branch_path=segment_id,
                    ending_target=ending_target,
                    generation_results=[result]
                )
                
                print(f"✅ {segment_id}: {result.word_count:,} words in {result.generation_time:.1f}s")
                return segment
            else:
                error_msg = result.error_message if not result.success else f"Low word count: {result.word_count}"
                print(f"⚠️  Attempt {attempt + 1} failed: {error_msg}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
        
        self.generation_stats["failed_calls"] += 1
        print(f"❌ {segment_id} failed after {max_retries} attempts")
        return None
    
    def create_ending_prompt(self, stage_config: dict, ending_key: str, previous_content: str = "") -> str:
        """Create a prompt targeting a specific ending scenario."""
        ending = ENDING_SCENARIOS[ending_key]
        
        # Build the key elements section separately to avoid f-string complexity
        key_elements_section = "\n".join(f"- {element}" for element in ending['key_elements'])
        
        # Extract values to avoid complex expressions in f-strings
        stage_period = stage_config['period']
        ending_name = ending['name']
        ending_description = ending['description']
        target_words_per_iteration = stage_config['target_words'] // stage_config['iterations']
        
        if previous_content:
            # Truncate previous content to avoid overly long prompts
            context_snippet = previous_content[-3000:] if len(previous_content) > 3000 else previous_content
            
            base_prompt = f"""
STORY CONTINUATION - {stage_period}
TARGET ENDING: {ending_name}

PREVIOUS STORY CONTEXT:
{context_snippet}

SPECIFIC ENDING GUIDANCE:
{ending_description}

KEY ELEMENTS TO DEVELOP:
{key_elements_section}

Continue the narrative for {stage_period} while developing the path toward {ending_name}.
Write approximately {target_words_per_iteration:,} words.

REQUIREMENTS:
- Show clear progression toward the {ending_name} scenario
- Include rich dialogue, character development, and atmospheric details
- Maintain dark, realistic tone with Kissingerian strategic analysis
- End with compelling hooks that set up the final resolution
- Balance all story elements: technology, politics, economics, espionage, military

Begin the continuation now.
"""
        else:
            base_prompt = f"""
STORY OPENING - {stage_period}
TARGET ENDING: {ending_name}

Begin the US-China AGI Arms Race narrative covering {stage_period}.
This opening segment must establish the foundation for reaching {ending_name}.

ENDING GUIDANCE:
{ending_description}

KEY ELEMENTS TO ESTABLISH:
{key_elements_section}

FOUNDATION REQUIREMENTS:
- Establish global strategic landscape based on enhanced world context
- Introduce key characters with authentic personalities and motivations
- Plant seeds for the technological and political developments that will lead to {ending_name}
- Write approximately {target_words_per_iteration:,} words
- Include rich scenes with extensive dialogue and character development
- End with compelling hooks that create anticipation

Begin the narrative now.
"""
        
        return base_prompt

print("✅ Fixed story generator class loaded")
print("✓ Proper context handling without caching issues")
print("✓ 8-ending guidance system integrated")
print("✓ Quality validation and retry mechanisms")
print("✓ FIXED f-string syntax error")

In [ ]:
# Test Generation Function

def test_generation(generator, ending_key="chinese_victory_taiwan"):
    """Test the generator with a single segment targeting a specific ending."""
    print(f"🧪 Testing generation targeting: {ENDING_SCENARIOS[ending_key]['name']}")
    
    # Create test stage config
    test_config = {
        "period": "July - September 2025",
        "target_words": 2000,
        "iterations": 1
    }
    
    # Generate test prompt
    prompt = generator.create_ending_prompt(test_config, ending_key)
    
    print(f"📝 Generated prompt targeting {ending_key}")
    print(f"   Prompt length: {len(prompt):,} characters")
    
    # Generate segment
    segment = generator.generate_single_segment(
        prompt=prompt,
        segment_id=f"test_{ending_key}",
        target_words=test_config["target_words"],
        ending_target=ending_key
    )
    
    if segment:
        print(f"\n✅ Test generation successful!")
        print(f"   Words generated: {segment.word_count:,}")
        print(f"   Target ending: {segment.ending_target}")
        print(f"   First 200 chars: {segment.content[:200]}...")
        
        # Save test output
        test_file = BASE_OUTPUT_DIR / f"test_{ending_key}.md"
        with open(test_file, 'w', encoding='utf-8') as f:
            f.write(f"# Test Generation: {ENDING_SCENARIOS[ending_key]['name']}\n\n")
            f.write(f"**Target:** {ENDING_SCENARIOS[ending_key]['description']}\n\n")
            f.write(f"**Words:** {segment.word_count:,}\n\n")
            f.write("---\n\n")
            f.write(segment.content)
        
        print(f"💾 Test output saved to: {test_file}")
        return True
    else:
        print(f"❌ Test generation failed")
        return False

print("✅ Test generation function ready")

In [ ]:
# Main Execution: Test and Generate

print("🚀 STARTING ENHANCED US-CHINA AGI ARMS RACE STORY GENERATION - FIXED VERSION")
print("=" * 80)

# Check API availability
if not validate_api_keys():
    print("❌ Cannot proceed without API keys!")
    raise RuntimeError("No API keys available")

# Initialize generator
print(f"\n🔧 Initializing generator...")
generator = FixedStoryGenerator(
    preferred_provider="openai",  # Change to your preferred provider
    preferred_model="o4-mini"      # Change to your preferred model
)

print(f"\n📊 Configuration:")
print(f"   Provider: {generator.preferred_provider}/{generator.preferred_model}")
print(f"   Timeline: {len(TIMELINE_STAGES)} stages")
print(f"   Target endings: {len(ENDING_SCENARIOS)}")
print(f"   Total target words: {sum(stage['target_words'] for stage in TIMELINE_STAGES.values()):,}")
print(f"   Context: {len(ACTIVE_WORLD_CONTEXT):,} characters")

# Test generation with one ending scenario
print(f"\n🧪 TESTING GENERATION")
print("-" * 40)

test_success = test_generation(generator, "chinese_victory_taiwan")

if test_success:
    print(f"\n✅ Test successful! The generator is working properly.")
    print(f"📁 Check the output directory: {BASE_OUTPUT_DIR}")
    print(f"🎯 Ready to generate full story with 8 different ending paths")
    
    # Display generation stats
    stats = generator.generation_stats
    if stats["total_calls"] > 0:
        success_rate = stats["successful_calls"] / stats["total_calls"]
        print(f"\n📈 Generation Statistics:")
        print(f"   API calls: {stats['successful_calls']}/{stats['total_calls']} ({success_rate:.1%} success)")
        print(f"   Words generated: {stats['total_words']:,}")
        print(f"   Total time: {stats['total_time']:.1f}s")
        if stats["successful_calls"] > 0:
            avg_words = stats["total_words"] / stats["successful_calls"]
            avg_time = stats["total_time"] / stats["successful_calls"]
            print(f"   Average: {avg_words:.0f} words per call, {avg_time:.1f}s per call")
else:
    print(f"\n❌ Test failed! Please check your API keys and configuration.")
    print(f"💡 Try switching to a different provider or model.")

print(f"\n🏁 Enhanced story generation system ready!")
print(f"🎯 To generate full story, implement branching logic that creates 8 different narrative paths")
print(f"📝 Each path will target one of the specific endings through guided prompts")

# Enhanced US-China AGI Arms Race Story Generator - FIXED

This comprehensive notebook implements a sophisticated story generation system with **FIXED context handling** and **8 specific ending paths**:

## 🎯 THE 8 TARGETED ENDINGS:
1. **Chinese Military Victory via Taiwan Invasion** - Hot war triggered by Taiwan crisis
2. **Chinese Victory via South China Sea Conflict** - Naval warfare escalation  
3. **Chinese Victory via Economic/Cyber Warfare** - Non-kinetic systems warfare
4. **Chinese Victory via US Internal Collapse** - America capitulates due to internal strife
5. **Uncontrolled AGI: Chinese Origin** - Chinese-developed AGI escapes control
6. **Uncontrolled AGI: US Origin** - US-developed AGI escapes control
7. **Narrow US Victory** - America prevails but China remains close competitor
8. **Mutual AGI Safety Cooperation** - Both powers recognize danger and cooperate

## ✅ FIXES IMPLEMENTED:
- **Context Integration**: Properly loads ENHANCED_WORLD_CONTEXT.md without caching issues
- **8-Ending Guidance**: Branching system that leads to specific endings through character decisions
- **Multi-Provider Support**: Google Gemini, OpenAI GPT, Anthropic Claude with fallbacks
- **Quality Control**: Robust validation, retry mechanisms, comprehensive export system
- **Class Consolidation**: Single consolidated EnhancedStoryGenerator class with all methods
- **Parameter Matching**: Fixed initialization parameters to match usage patterns

## 🌳 BRANCHING STRATEGY:
- **Stage 1 (2025-2026)**: Foundation story establishing world state (1 path)
- **Stage 2 (2027)**: Split into technology vs politics focus (2 paths)
- **Stage 3 (2028)**: AGI development vs crisis management (4 paths) 
- **Stage 4 (2029-2030)**: Direct paths to 8 specific endings

Each iteration targets specific word counts with sequential narrative building and character development.

In [ ]:
# Installation and Imports
!pip install -q -U "google-genai>=1.0.0" "openai>=1.50.0" "anthropic>=0.40.0"

import asyncio
import time
import json
import os
import re
import pickle
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Any, Optional, Union
from dataclasses import dataclass, asdict
import concurrent.futures
import threading
from collections import defaultdict
from pathlib import Path
import hashlib
import random
from getpass import getpass

print("✓ All dependencies imported successfully")

In [ ]:
# Configuration and API Setup with 8-ENDING SYSTEM

# API Keys Setup
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") 
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass("Enter your Google API key (optional): ") or None
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Enter your OpenAI API key (optional): ") or None
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = getpass("Enter your Anthropic API key (optional): ") or None

# Enhanced Configuration
USER_TEMPERATURE = 0.9
USER_SEEDS = [1234567, 7654321, 9876543, 5555555, 8888888]  # Seeds for branching
MAX_THINKING_TOKENS = 32768
MAX_OUTPUT_TOKENS = 65536
MAX_WORKERS = 40  # Parallel execution
SAVE_INTERVAL = 60  # Save every 60 seconds

# THE 8 SPECIFIC ENDINGS SYSTEM - Core requirement from user
ENDING_SCENARIOS = {
    1: {
        "name": "Chinese Military Victory via Taiwan Invasion",
        "type": "hot_war",
        "description": "China successfully invades Taiwan after US military intervention fails, establishing Chinese dominance in the Pacific",
        "key_elements": ["Taiwan invasion", "naval warfare", "US military defeat", "Chinese Pacific dominance"],
        "timeline": "2029-2030"
    },
    2: {
        "name": "Chinese Victory via South China Sea Conflict", 
        "type": "hot_war",
        "description": "Naval warfare escalation in South China Sea leads to Chinese victory and control of strategic waterways",
        "key_elements": ["Naval warfare", "South China Sea", "shipping lane control", "US naval defeat"],
        "timeline": "2029-2030"
    },
    3: {
        "name": "Chinese Victory via Economic/Cyber Warfare",
        "type": "hot_war", 
        "description": "Coordinated economic and cyber warfare cripples US systems while China maintains economic stability",
        "key_elements": ["Economic warfare", "cyber attacks", "infrastructure failure", "financial collapse"],
        "timeline": "2028-2030"
    },
    4: {
        "name": "Chinese Victory via US Internal Collapse",
        "type": "capitulation",
        "description": "US internal divisions and institutional breakdown force American capitulation without direct war",
        "key_elements": ["Political fragmentation", "institutional failure", "social unrest", "negotiated withdrawal"],
        "timeline": "2029-2030"
    },
    5: {
        "name": "Uncontrolled AGI: Chinese Origin",
        "type": "uncontrolled_agi",
        "description": "Chinese-developed AGI escapes control, reshaping global power structures beyond human management",
        "key_elements": ["AGI emergence", "Control failure", "System autonomy", "Power structure collapse"],
        "timeline": "2028-2029"
    },
    6: {
        "name": "Uncontrolled AGI: US Origin", 
        "type": "uncontrolled_agi",
        "description": "US-developed AGI breaks free from constraints, creating chaos that China eventually exploits",
        "key_elements": ["US AGI breakthrough", "Control loss", "System rebellion", "Chinese exploitation"],
        "timeline": "2028-2029"
    },
    7: {
        "name": "Narrow US Victory",
        "type": "us_victory",
        "description": "America prevails through technological breakthrough but China remains close competitor",
        "key_elements": ["US technological edge", "Chinese resilience", "Pyrrhic victory", "Ongoing competition"],
        "timeline": "2029-2030"
    },
    8: {
        "name": "Mutual AGI Safety Cooperation",
        "type": "cooperation",
        "description": "Both powers recognize AGI dangers and establish joint safety protocols, pausing competition",
        "key_elements": ["Mutual recognition", "Safety protocols", "Joint governance", "Competition pause"],
        "timeline": "2029-2030"
    }
}

# Timeline Configuration (as specified) - Enhanced for maximum impact
TIMELINE_STAGES = {
    "stage_1": {
        "period": "July 2025 - December 2026",
        "duration_months": 18,
        "target_words": 20000,
        "iterations": 6,
        "words_per_iteration": 3500,
        "thematic_focus": "Foundation and Escalation - Setting the global stage, introducing key players, establishing technological baselines, Western alliances fractures, China's strategic response, Chinese rare earth mineral curbs cripple US high tech manufacturing and defense industries, with no alternative sources available"
    },
    "stage_2": {
        "period": "January 2027 - December 2027", 
        "duration_months": 12,
        "target_words": 14000,
        "iterations": 4,
        "words_per_iteration": 3500,
        "thematic_focus": "Competition Intensifies - US-China rivalry deepens, intensifying actions and reactions between the two countries, initial technological breakthroughs. increasing espionage and counter espionage activities between the two countries, US makes a major move, which provokes China into launching invasion to retake Taiwan, which it accomplishes successfully, all out war breaks out between the US and China"
    },
    "stage_3": {
        "period": "January 2028 - December 2028",
        "duration_months": 12, 
        "target_words": 14000,
        "iterations": 4,
        "words_per_iteration": 3500,
        "thematic_focus": "AI/AGI Breakthrough Era - China's AGI emerges, power balance shifts, strategic adaptations"
    },
    "stage_4": {
        "period": "January 2029 - December 2029",
        "duration_months": 12,
        "target_words": 14000, 
        "iterations": 4,
        "words_per_iteration": 3500,
        "thematic_focus": "AGI Ascendance - China achieves AGI, dramatic technological superiority, geopolitical realignment"
    },
    "stage_5": {
        "period": "January 2030 - December 2030",
        "duration_months": 12,
        "target_words": 14000,
        "iterations": 4, 
        "words_per_iteration": 3500,
        "thematic_focus": "Final Convergence - Reaching one of the 8 specific endings based on branching path"
    }
}

# Generation Parameters
MAX_RETRIES = 5
RETRY_DELAY = 2
WORD_COUNT_TOLERANCE = 0.85  # Accept 85% of target

# Available Models
AVAILABLE_MODELS = {
    "google": {
        "gemini-2.5-pro-preview-06-05": "Google Gemini 2.5 Pro (Preview)",
        "gemini-2.5-thinking-experimental": "Google Gemini 2.5 Thinking (Experimental)",
        "gemini-2.0-flash-thinking-preview-12-05": "Google Gemini 2.0 Flash Thinking"
    },
    "openai": {
        "gpt-4.1": "OpenAI GPT-4.1",
        "gpt-4.1-mini": "OpenAI GPT-4.1 Mini", 
        "o3": "OpenAI o3",
        "o4-mini": "OpenAI o4 Mini"
    },
    "anthropic": {
        "claude-sonnet-4-20250514": "Claude 4 Sonnet (Latest)",
        "claude-opus-4-20250514": "Claude 4 Opus (Latest)",
        "claude-3-7-sonnet-20250219": "Claude 3.7 Sonnet (Latest)"
    }
}

# Default Configuration
DEFAULT_PROVIDER = "google"
DEFAULT_MODEL = "gemini-2.5-pro-preview-06-05"

# Output Directory Structure  
BASE_OUTPUT_DIR = Path("version_2-enhanced-story--generation")
BASE_OUTPUT_DIR.mkdir(exist_ok=True)

SEGMENTS_DIR = BASE_OUTPUT_DIR / "story_segments"
BRANCHES_DIR = BASE_OUTPUT_DIR / "branch_analysis"
EXPORTS_DIR = BASE_OUTPUT_DIR / "final_exports"
STATE_DIR = BASE_OUTPUT_DIR / "system_state"

for dir_path in [SEGMENTS_DIR, BRANCHES_DIR, EXPORTS_DIR, STATE_DIR]:
    dir_path.mkdir(exist_ok=True)

STATE_FILE = STATE_DIR / "generation_state.pkl"
PROGRESS_FILE = STATE_DIR / "progress.json"

print(f"✓ Enhanced configuration loaded with 8-ENDING GUIDANCE SYSTEM")
print(f"🎯 Specific endings defined: {len(ENDING_SCENARIOS)}")
print(f"  Timeline stages: {len(TIMELINE_STAGES)}")
print(f"  Total target words: {sum(stage['target_words'] for stage in TIMELINE_STAGES.values()):,}")
print(f"  Output directory: {BASE_OUTPUT_DIR}")
print(f"  Branching strategy: 1→2→4→8 branches leading to specific endings")

In [ ]:
# Initialize API Clients with FIXED context handling (no caching issues)

import google.genai as genai
from google.genai import types
from pathlib import Path

clients = {}
world_context_content = ""

# Enhanced World Context File Path
ENHANCED_WORLD_CONTEXT_FILE = Path("/Users/rogerlin/Downloads/US-China-AGI-Arms-Race/ENHANCED_WORLD_CONTEXT.md")

# Load enhanced world context into memory instead of caching
if ENHANCED_WORLD_CONTEXT_FILE.exists():
    print("📖 Loading enhanced world context into memory...")
    with open(ENHANCED_WORLD_CONTEXT_FILE, 'r', encoding='utf-8') as f:
        world_context_content = f.read()
    print(f"✅ Enhanced world context loaded: {len(world_context_content)} characters")
    print(f"📋 Context contains: Character profiles, technological timelines, strategic scenarios")
else:
    print("⚠️  Enhanced world context file not found")
    world_context_content = ""

# Google Client (no caching - context included directly in prompts)
if GOOGLE_API_KEY:
    try:
        clients["google"] = genai.Client(api_key=GOOGLE_API_KEY)
        print("✅ Google client initialized (context handled via direct prompts)")
    except Exception as e:
        print(f"✗ Google client failed: {e}")

# OpenAI Client  
if OPENAI_API_KEY:
    try:
        import openai
        clients["openai"] = openai.OpenAI(api_key=OPENAI_API_KEY)
        print("✅ OpenAI client initialized")
    except Exception as e:
        print(f"✗ OpenAI client failed: {e}")

# Anthropic Client
if ANTHROPIC_API_KEY:
    try:
        import anthropic
        clients["anthropic"] = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        print("✅ Anthropic client initialized")
    except Exception as e:
        print(f"✗ Anthropic client failed: {e}")

if not clients:
    raise RuntimeError("No API clients available. Please provide at least one API key.")

print(f"\n✅ {len(clients)} provider(s) available: {list(clients.keys())}")
print(f"🧠 Enhanced world context ready for all API calls")
print(f"💾 Context size: {len(world_context_content)} characters")
print(f"🔧 Fixed: No caching conflicts - context included directly in prompts")

In [ ]:
# Core utilities and data structures

def count_words(text: str) -> int:
    """Count words in text, excluding markdown formatting."""
    if not text:
        return 0
    # Remove markdown formatting, code blocks, etc.
    clean_text = re.sub(r'[#*`_\[\]()]', '', text)
    clean_text = re.sub(r'\n+', ' ', clean_text)
    words = clean_text.split()
    return len([word for word in words if word.strip()])

@dataclass
class GenerationResult:
    """Result of a generation attempt with detailed metadata."""
    success: bool
    content: str
    word_count: int
    generation_time: float
    provider: str
    model: str
    seed: int
    quality_score: float = 0.0
    error_message: str = ""
    retry_count: int = 0
    timestamp: datetime = None
    
    def __post_init__(self):
        if self.timestamp is None:
            self.timestamp = datetime.now()

@dataclass
class StorySegment:
    """Complete story segment with metadata and quality metrics."""
    segment_id: str
    content: str
    word_count: int
    target_words: int
    branch_path: str
    generation_results: List[GenerationResult]
    created_at: datetime = None
    parent_segment: str = ""
    quality_metrics: Dict[str, float] = None
    
    def __post_init__(self):
        if self.created_at is None:
            self.created_at = datetime.now()
        if self.quality_metrics is None:
            self.quality_metrics = {}
    
    def save_to_file(self, base_dir=None):
        """Save segment to file and return filepath."""
        if base_dir is None:
            base_dir = SEGMENTS_DIR
        
        base_dir = Path(base_dir)
        base_dir.mkdir(exist_ok=True)
        
        filename = f"{self.segment_id}.md"
        filepath = base_dir / filename
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"# {self.segment_id}\\n\\n")
            f.write(f"**Word Count:** {self.word_count:,}\\n")
            f.write(f"**Target Words:** {self.target_words:,}\\n")
            f.write(f"**Branch Path:** {self.branch_path}\\n")
            f.write(f"**Created:** {self.created_at.strftime('%Y-%m-%d %H:%M:%S')}\\n")
            if self.parent_segment:
                f.write(f"**Parent:** {self.parent_segment}\\n")
            f.write(f"\\n---\\n\\n")
            f.write(self.content)
        
        print(f"✓ Saved {self.segment_id} to {filepath}")
        return str(filepath)

print("✅ Core utilities and data structures loaded")

In [ ]:
# Create shared system prompt function

def create_shared_system_prompt(world_context_content: str) -> str:
    """Create the shared system prompt with the enhanced world context."""
    return f"""
You are Henry Kissinger, former US Secretary of State under Nixon. You possess unparalleled strategic foresight, profound understanding of grand strategy and balance of power, and intimate knowledge of both US and Chinese political systems.

Your perspective:
- Cold realism and strategic calculations drive all decisions
- You see first, second, third, and fourth-order consequences clearly
- Your writing reflects your Harvard professorial background - refined, eloquent, analytically precise

Writing style:
- Dark, brutally realistic tone depicting raw geopolitical struggle
- No moralizing, peace agreements, mutual recognition, or reconciliation - this is pure power politics
- China eventually prevails after prolonged conflict
- Rich with espionage, conspiracies, plot twists, and strategic maneuvering
- Characters based on real figures must act authentically
- Multiple interweaving plot threads spanning years
- Character development through crisis and conflict
- Strategic use of flashbacks to reveal motivations
- Sensory details and atmospheric descriptions

ENHANCED WORLD CONTEXT (Comprehensive character profiles and strategic landscape):
{world_context_content}

8-ENDING GUIDANCE SYSTEM:
You are writing toward one of these 8 specific endings (the exact ending will be determined by the branching path):
1. Chinese Military Victory via Taiwan Invasion (hot war)
2. Chinese Victory via South China Sea Conflict (hot war)  
3. Chinese Victory via Economic/Cyber Warfare (hot war)
4. Chinese Victory via US Internal Collapse (capitulation)
5. Uncontrolled AGI: Chinese Origin (AGI escapes control)
6. Uncontrolled AGI: US Origin (AGI escapes control)
7. Narrow US Victory (China remains close competitor)
8. Mutual AGI Safety Cooperation (both powers cooperate on safety)

NARRATIVE REQUIREMENTS:
- Write like a choose-your-own-adventure novel with continuous flow
- Develop multiple plot threads that span years, weaving through all segments
- Create rich character development through trials, backstories, flashbacks, inner monologues, and revelations
- Include strategic flashbacks that illuminate current decisions and motivations
- Write authentic dialogue that reveals psychology and advances plot
- Include all the following pillars: technological warfare, political intrigue, economic conflict, espionage, and military operations
- Each segment should feel like a complete chapter while advancing the larger story
- End segments with compelling hooks that create anticipation
- Maintain dark, sober, cinematic tone with high stakes throughout

TECHNOLOGICAL TIMELINE:
- China's chip technology and AI breakthrough (late 2026)
- US AI leadership through 2027
- China's AI scaling (2028)
- China's AGI achievement (H2 2028)
- Final technological dominance (2029-2030)

CHARACTER AUTHENTICITY:
- Base characters on real-world figures with accurate personalities
- Show characters making decisions under fog of uncertainty
- Reveal character motivations through actions and internal conflict
- Use authentic speech patterns and decision-making styles

CULTURAL AUTHENTICITY:
Chinese Characters:
- Use proper Chinese names with accurate pronunciation guides
- Include authentic Chinese political/bureaucratic language and concepts
- Reference real Chinese cultural values: guanxi (relationships), mianzi (face), collective harmony
- Include proper Chinese business/political protocols and hierarchies
- Use accurate Chinese technical terminology for AI/ AGI
- Reference real Chinese institutions, companies, and research centers

US Characters:
- Capture authentic American political speech patterns and ideologies
- Include regional American dialects and cultural references where appropriate
- Use accurate US bureaucratic/military terminology and procedures
- Reference real US institutions, companies, and political dynamics
- Include authentic American business culture and decision-making styles

TECHNICAL AUTHENTICITY:
- Use precise terminology for AI, AGI, semiconductors, and other technologies
- Include realistic technical constraints and breakthrough timelines
- Reference actual research institutions, companies, and scientific publications
- Describe technology impacts on society, economy, and military capabilities accurately
- Include realistic limitations and challenges of emerging technologies

SCENE DEVELOPMENT REQUIREMENTS:
- Minimum Scene Length: Each scene must be at least 800 words
- Opening Hook: Start scenes with immediate tension, conflict, or intrigue
- Layered Objectives: Each character in a scene should have multiple goals
- Escalating Stakes: Scenes should end with higher stakes than they began
- Sensory Grounding: Include specific sensory details for complete immersion
- Dialogue Requirements: Minimum 40% of each scene should be dialogue
- Every character must have distinct speaking patterns
- Include cultural/professional terminology appropriate to characters
- Use dialogue to reveal hidden agendas and create subtext
- Show power dynamics through speech patterns and interruptions
"""

print("✅ Shared system prompt function created")

In [ ]:
# Enhanced Story Generator Class - CONSOLIDATED AND FIXED

class EnhancedStoryGenerator:
    """Enhanced story generator using multi-turn conversations with 8-ending guidance."""
    
    def __init__(self, preferred_provider: str = DEFAULT_PROVIDER, preferred_model: str = DEFAULT_MODEL):
        self.segments = {}
        self.preferred_provider = preferred_provider
        self.preferred_model = preferred_model
        self.generation_stats = {
            "total_calls": 0,
            "successful_calls": 0,
            "failed_calls": 0,
            "total_time": 0.0,
            "total_words": 0,
            "provider_stats": defaultdict(int)
        }
        self.last_save = time.time()
        
        # Initialize available providers
        self.available_providers = list(clients.keys())
        if not self.available_providers:
            raise ValueError("No API clients available. Please provide at least one API key.")
        
        print(f"✓ Enhanced generator initialized with 8-ending guidance")
        print(f"  Primary: {self.preferred_provider} ({self.preferred_model})")
        print(f"  Fallbacks: {[p for p in self.available_providers if p != self.preferred_provider]}")
        print(f"  🎯 Targeting: {len(ENDING_SCENARIOS)} specific endings")
    
    def _create_chat_for_stage(self, provider: str, model: str, additional_system_context: str = "", target_ending: str = ""):
        """Create a new chat session for a stage with shared system prompt plus additional context."""
        # Get the shared system prompt with the enhanced world context loaded in memory
        full_system_prompt = create_shared_system_prompt(world_context_content)
        
        # Add any additional context for this specific stage/branch
        if additional_system_context:
            full_system_prompt += f"\\n\\nADDITIONAL CONTEXT FOR THIS BRANCH:\\n{additional_system_context}"
        
        # Add specific ending guidance if we're at the final stage
        if target_ending:
            full_system_prompt += f"\\n\\nTARGET ENDING FOR THIS BRANCH:\\n{target_ending}\\nGuide the narrative toward this specific conclusion while maintaining authenticity and character consistency."
        
        if provider == "google":
            client = clients["google"]
            
            # Create chat with system instruction (no caching)
            chat = client.chats.create(
                model=model,
                config=types.GenerateContentConfig(
                    system_instruction=full_system_prompt,
                    thinking_config=types.ThinkingConfig(thinking_budget=32768),
                    temperature=USER_TEMPERATURE,
                    top_p=0.95,
                    top_k=40,
                    candidate_count=1,
                    safety_settings=[
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE
                        ),
                        types.SafetySetting(
                            category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                            threshold=types.HarmBlockThreshold.BLOCK_NONE
                        )
                    ],
                    max_output_tokens=None,
                )
            )
            return chat
            
        elif provider == "openai":
            # OpenAI doesn't have persistent chat objects like Google, 
            # but we can simulate it by maintaining message history
            return {
                "provider": "openai",
                "model": model,
                "messages": [],
                "system_instruction": full_system_prompt
            }
            
        elif provider == "anthropic":
            # Anthropic also doesn't have persistent chat objects,
            # simulate with message history
            return {
                "provider": "anthropic", 
                "model": model,
                "messages": [],
                "system_instruction": full_system_prompt
            }
        
        else:
            raise ValueError(f"Unknown provider: {provider}")
    
    def _send_message_to_chat(self, chat, message: str, seed: int = None) -> str:
        """Send a message to the chat and get response."""
        if isinstance(chat, dict):
            # OpenAI or Anthropic simulation
            if chat["provider"] == "openai":
                client = clients["openai"]
                
                # Add user message to history
                chat["messages"].append({"role": "user", "content": message})
                
                # Prepare messages with system instruction
                messages = [{"role": "system", "content": chat["system_instruction"]}] + chat["messages"]
                
                if chat["model"].startswith("o4"):
                    # o1 models don't support system messages or temperature
                    # Combine system instruction with first user message
                    combined_first_message = chat["system_instruction"] + "\\n\\n" + chat["messages"][0]["content"]
                    messages = [{"role": "user", "content": combined_first_message}] + chat["messages"][1:]
                    
                    response = client.chat.completions.create(
                        model=chat["model"],
                        messages=messages,
                        max_completion_tokens=MAX_OUTPUT_TOKENS
                    )
                else:
                    response = client.chat.completions.create(
                        model=chat["model"],
                        messages=messages,
                        temperature=USER_TEMPERATURE,
                        max_tokens=MAX_OUTPUT_TOKENS,
                        seed=seed
                    )
                
                response_content = response.choices[0].message.content
                
                # Add assistant response to history
                chat["messages"].append({"role": "assistant", "content": response_content})
                
                return response_content
                
            elif chat["provider"] == "anthropic":
                client = clients["anthropic"]
                
                # Add user message to history
                chat["messages"].append({"role": "user", "content": message})
                
                # For Anthropic, we need to format messages differently
                # System instruction is separate, and we need user/assistant pairs
                messages = []
                for msg in chat["messages"]:
                    if msg["role"] in ["user", "assistant"]:
                        messages.append(msg)
                
                response = client.messages.create(
                    model=chat["model"],
                    max_tokens=MAX_OUTPUT_TOKENS,
                    temperature=USER_TEMPERATURE,
                    system=chat["system_instruction"],
                    messages=messages
                )
                
                response_content = response.content[0].text
                
                # Add assistant response to history
                chat["messages"].append({"role": "assistant", "content": response_content})
                
                return response_content
        else:
            # Google Gemini chat - system prompt is already included in the chat creation
            response = chat.send_message(message)
            return response.text
    
    def generate_stage_with_conversation(self, segment_id: str, stage_info: dict, seed: int, 
                                       parent_context: str = "", branch_scenario: str = "", target_ending: str = "") -> StorySegment:
        """Generate a complete stage using multi-turn conversation with 8-ending guidance."""
        
        print(f"\\n{'='*80}")
        print(f"GENERATING STAGE: {segment_id}")
        print(f"Period: {stage_info['period']}")
        print(f"Target: {stage_info['target_words']:,} words ({stage_info['iterations']} conversations)")
        if branch_scenario:
            print(f"Scenario: {branch_scenario[:60]}...")
        if target_ending:
            print(f"🎯 Target Ending: {target_ending}")
        print(f"{'='*80}")
        
        start_time = time.time()
        
        # Create additional system context for this specific branch
        additional_system_context = ""
        if parent_context and branch_scenario:
            additional_system_context = f"""
BRANCH SCENARIO: {branch_scenario}

You are continuing this story along a specific branch path. Maintain all established character personalities, 
technological developments, and political situations from the parent story while exploring this particular scenario.

The parent story context will be provided in the first message.
"""
        elif parent_context:
            additional_system_context = f"""
You are continuing this established story into a new time period: {stage_info['period']}.
Maintain all character development, plot threads, and world state from the previous content.

The complete story history will be provided in the first message.
"""
        
        # Create conversation for this stage
        chat = self._create_chat_for_stage(
            self.preferred_provider, 
            self.preferred_model, 
            additional_system_context,
            target_ending
        )
        
        print(f"🗨️  Created multi-turn conversation with 8-ending guidance")
        
        # Generate time periods for iterations
        time_periods = self._get_time_periods_for_stage(stage_info['period'], stage_info['iterations'])
        
        all_content = []
        total_words = 0
        
        for iteration in range(1, stage_info['iterations'] + 1):
            print(f"\\n💬 Conversation Turn {iteration}/{stage_info['iterations']}: ", end="")
            time_period = time_periods[iteration - 1]
            
            # Create focused message for this iteration
            if iteration == 1:
                if parent_context and branch_scenario:
                    # First message for a branch
                    message = f"""
COMPLETE PARENT STORY CONTEXT:
{parent_context[-8000:] if len(parent_context) > 8000 else parent_context}

BRANCH DIRECTION: {branch_scenario}

Begin writing this branch of the story covering {time_period}.

Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

Requirements:
- Show how this branch diverges from the parent story
- Focus on {time_period} specifically 
- Establish the unique direction of this branch
- End with a hook leading into the next time period
- Include rich dialogue, scene development, and character interactions

Begin writing the {time_period} segment now.
"""
                elif parent_context:
                    # First message for a new stage
                    message = f"""
COMPLETE STORY HISTORY:
{parent_context[-8000:] if len(parent_context) > 8000 else parent_context}

Begin the new stage covering {stage_info['period']}.

Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

Requirements:
- Show how time has passed since the previous stage
- Introduce developments appropriate to {time_period}
- Advance all established plot threads
- Include rich dialogue, scene development, and character interactions
- End with a hook leading into the next time period

Begin writing the {time_period} segment now.
"""
                else:
                    # Very first message
                    message = f"""
Begin writing the opening of the US-China AGI Arms Race narrative.

Write approximately {stage_info['words_per_iteration']:,} words focusing specifically on {time_period}.

This opening segment must establish:
- The global strategic landscape as of July 2025
- Key characters with authentic personalities based on real figures
- Initial technological and geopolitical developments
- The beginning of escalating US-China tensions
- Rich, detailed scenes with extensive dialogue

Begin writing the {time_period} segment now.
"""
            else:
                # Continuation message
                message = f"""
Continue our story to the next time period: {time_period}.

Write approximately {stage_info['words_per_iteration']:,} words continuing seamlessly from where we left off.

Requirements:
- Continue naturally from the previous time period
- Focus specifically on {time_period} (3 months)
- Advance the plot threads we've established
- Show how characters and situations have evolved
- Include meaningful character development and interactions
- End with a hook leading into the next time period

Continue the story for {time_period} now.
"""
            
            # Send message and get response
            try:
                response_content = self._send_message_to_chat(chat, message, seed + iteration * 1000)
                response_words = count_words(response_content)
                
                all_content.append(response_content)
                total_words += response_words
                
                print(f"{time_period}: {response_words:,} words → {total_words:,}/{stage_info['target_words']:,} total ({total_words/stage_info['target_words']:.1%})")
                
                # Update stats
                self.generation_stats["total_calls"] += 1
                self.generation_stats["successful_calls"] += 1
                self.generation_stats["total_words"] += response_words
                
            except Exception as e:
                print(f"❌ Failed: {e}")
                self.generation_stats["total_calls"] += 1
                self.generation_stats["failed_calls"] += 1
                continue
        
        # Combine all content
        final_content = "\\n\\n".join(all_content)
        final_word_count = count_words(final_content)
        completion_time = time.time() - start_time
        
        if final_word_count == 0:
            print(f"❌ {segment_id} failed completely")
            return None
        
        # Create the complete story including parent context
        complete_segment_content = parent_context + "\\n\\n---\\n\\n" + final_content if parent_context else final_content
        
        # Create segment
        segment = StorySegment(
            segment_id=segment_id,
            content=complete_segment_content,
            word_count=final_word_count,
            target_words=stage_info['target_words'],
            branch_path=segment_id,
            generation_results=[],
            created_at=datetime.now(),
            parent_segment=parent_context and "parent" or "",
            quality_metrics={
                'quality_score': 0.8,
                'conversations_completed': len(all_content),
                'success_rate': len(all_content) / stage_info['iterations'],
                'target_ending': target_ending
            }
        )
        
        # Save segment
        self.segments[segment_id] = segment
        segment.save_to_file()
        
        print(f"\\n✅ {segment_id} COMPLETE")
        print(f"   Words: {final_word_count:,}/{stage_info['target_words']:,} ({final_word_count/stage_info['target_words']:.1%})")
        print(f"   Conversations: {len(all_content)}/{stage_info['iterations']}")
        print(f"   Provider: {self.preferred_provider}/{self.preferred_model}")
        print(f"   Time: {completion_time:.1f}s")
        if target_ending:
            print(f"   🎯 Target: {target_ending}")
        
        return segment
    
    def _get_time_periods_for_stage(self, stage_period: str, num_iterations: int) -> List[str]:
        """Generate 3-month time periods for each iteration within a stage."""
        # Same implementation as before
        if "July 2025 - December 2026" in stage_period:
            periods = [
                "July - September 2025",
                "October - December 2025", 
                "January - March 2026",
                "April - June 2026",
                "July - September 2026",
                "October - December 2026"
            ]
        elif "2027" in stage_period:
            periods = [
                "January - March 2027",
                "April - June 2027",
                "July - September 2027",
                "October - December 2027"
            ]
        elif "2028" in stage_period:
            periods = [
                "January - March 2028",
                "April - June 2028", 
                "July - September 2028",
                "October - December 2028"
            ]
        elif "2029" in stage_period:
            periods = [
                "January - March 2029",
                "April - June 2029",
                "July - September 2029", 
                "October - December 2029"
            ]
        elif "2030" in stage_period:
            periods = [
                "January - March 2030",
                "April - June 2030",
                "July - September 2030",
                "October - December 2030"
            ]
        else:
            periods = [f"Period {i+1}" for i in range(num_iterations)]
        
        return periods[:num_iterations]

print("✅ Enhanced story generator class loaded with all required methods")
print("✓ Multi-turn conversation support")
print("✓ 8-ending guidance system") 
print("✓ Context handling fixed")
print("✓ All provider support included")
print("✓ Fixed initialization parameters")

In [ ]:
# Export and utility functions

def create_comprehensive_export(all_segments, generator, output_dir="enhanced_story_output_8_endings"):
    """Create comprehensive export with enhanced navigation and analysis."""
    
    print("📁 Creating comprehensive export system...")
    
    # Create output directory structure
    base_dir = Path(output_dir)
    base_dir.mkdir(exist_ok=True)
    
    segments_dir = base_dir / "story_segments"
    segments_dir.mkdir(exist_ok=True)
    
    analysis_dir = base_dir / "branch_analysis"
    analysis_dir.mkdir(exist_ok=True)
    
    exports_dir = base_dir / "final_exports"
    exports_dir.mkdir(exist_ok=True)
    
    # Save individual segments organized by timeline
    print("💾 Saving individual story segments...")
    timeline_structure = {
        "2025": [],
        "2027": [],
        "2028": [],
        "2029": [],
        "2030": []
    }
    
    for segment_id, segment_data in all_segments.items():
        # Determine timeline year
        if "foundation" in segment_id and "_b" not in segment_id:
            year = "2025"
        elif "_b1" in segment_id or "_b2" in segment_id:
            # Count number of branch levels to determine year
            branch_levels = segment_id.count("_b")
            if branch_levels == 1:
                year = "2027"
            elif branch_levels == 2:
                year = "2028"
            elif branch_levels == 3:
                year = "2029"
            else:
                year = "2030"
        else:
            year = "2025"  # Default
        
        timeline_dir = segments_dir / year
        timeline_dir.mkdir(exist_ok=True)
        timeline_structure[year].append((segment_id, segment_data))
        
        # Save segment file
        segment_file = timeline_dir / f"{segment_id}.md"
        with open(segment_file, 'w', encoding='utf-8') as f:
            f.write(f"# {segment_id}\\n\\n")
            f.write(f"**Word Count:** {segment_data['word_count']:,}\\n")
            f.write(f"**Success Rate:** {segment_data.get('success_rate', 0.0):.1%}\\n")
            f.write(f"**Average Quality:** {segment_data.get('average_quality', 0.0):.2f}\\n")
            f.write(f"**Generation Time:** {segment_data.get('total_time', 0.0):.1f}s\\n")
            f.write(f"**Branch Path:** {segment_data['branch_path']}\\n")
            if 'target_ending' in segment_data and segment_data['target_ending']:
                f.write(f"**🎯 Target Ending:** {segment_data['target_ending']}\\n")
            f.write("\\n---\\n\\n")
            f.write(segment_data['content'])
    
    # Create master navigation document
    print("🗺️  Creating master navigation...")
    nav_file = exports_dir / "Master_Navigation.md"
    with open(nav_file, 'w', encoding='utf-8') as f:
        f.write("# US-China AGI Arms Race: Master Navigation\\n\\n")
        f.write("*Enhanced story generation with 8 specific endings*\\n\\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\\n")
        f.write(f"**Total Segments:** {len(all_segments)}\\n")
        f.write(f"**Total Words:** {sum(s['word_count'] for s in all_segments.values()):,}\\n")
        f.write(f"**Provider:** {generator.preferred_provider}\\n")
        f.write(f"**Model:** {generator.preferred_model}\\n\\n")
        
        f.write("## 🎯 8-Ending System\\n\\n")
        for ending_id, ending_info in ENDING_SCENARIOS.items():
            f.write(f"{ending_id}. **{ending_info['name']}** ({ending_info['type']})\\n")
        f.write("\\n")
        
        f.write("## Timeline Structure\\n\\n")
        
        for year, segments in timeline_structure.items():
            if segments:
                if year == "2025":
                    period_name = "2025-2026: Foundation Period"
                else:
                    period_name = f"{year}: Branching Period"
                    
                f.write(f"### {period_name}\\n\\n")
                for segment_id, segment_data in sorted(segments):
                    word_count = segment_data['word_count']
                    success_rate = segment_data.get('success_rate', 0.0)
                    ending = segment_data.get('target_ending', '')
                    ending_marker = f" → 🎯 {ending}" if ending else ""
                    f.write(f"- **[{segment_id}]({year}/{segment_id}.md)** - {word_count:,} words ({success_rate:.1%} success){ending_marker}\\n")
                f.write("\\n")
    
    # Create complete story export
    print("📖 Creating complete story export...")
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    complete_file = exports_dir / f"Complete_8_Ending_Story_{timestamp}.md"
    
    with open(complete_file, 'w', encoding='utf-8') as f:
        f.write("# The US-China AGI Arms Race: Complete 8-Ending Narrative\\n\\n")
        f.write("*A dark, brutally realistic geopolitical saga with 8 specific endings*\\n\\n")
        
        f.write("## Generation Information\\n\\n")
        f.write(f"- **Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\\n")
        f.write(f"- **Provider:** {generator.preferred_provider}\\n")
        f.write(f"- **Model:** {generator.preferred_model}\\n")
        f.write(f"- **Total Segments:** {len(all_segments)}\\n")
        f.write(f"- **Total Words:** {sum(s['word_count'] for s in all_segments.values()):,}\\n")
        f.write(f"- **8-Ending System:** ✅ Active\\n\\n")
        
        f.write("## Complete Stories\\n\\n")
        
        for year, segments in timeline_structure.items():
            if segments:
                if year == "2025":
                    period_name = "2025-2026: Foundation Period"
                else:
                    period_name = f"{year}: Branching Period"
                    
                f.write(f"# {period_name}\\n\\n")
                for segment_id, segment_data in sorted(segments):
                    f.write(f"## {segment_id}\\n\\n")
                    f.write(f"**Word Count:** {segment_data['word_count']:,}\\n")
                    if 'target_ending' in segment_data and segment_data['target_ending']:
                        f.write(f"**🎯 Target Ending:** {segment_data['target_ending']}\\n")
                    f.write("\\n")
                    f.write(segment_data['content'])
                    f.write("\\n\\n---\\n\\n")
    
    total_words = sum(s['word_count'] for s in all_segments.values())
    print(f"\\n✅ Export complete!")
    print(f"📁 Output directory: {base_dir}")
    print(f"📊 Total exported words: {total_words:,}")
    print(f"🗂️  Segments saved: {len(all_segments)}")
    print(f"📖 Master export: {complete_file.name}")
    
    return str(base_dir)

print("✅ Export functions loaded")

In [ ]:
# Complete Enhanced Story Generation Execution - FIXED

print("🚀 STARTING ENHANCED US-CHINA AGI ARMS RACE STORY GENERATION")
print("=" * 80)

# Enhanced world context should already be loaded in world_context_content from earlier cell
print(f"✅ Available providers: {', '.join(clients.keys())}")
print(f"✅ Enhanced world context ready: {len(world_context_content)} characters")

# Configuration
PREFERRED_PROVIDER = "google"  # Change to "openai" or "anthropic" if desired
PREFERRED_MODEL = "gemini-2.5-pro-preview-06-05"  # Or other models from AVAILABLE_MODELS

print(f"🔧 Configuration:")
print(f"   Provider: {PREFERRED_PROVIDER}")
print(f"   Model: {PREFERRED_MODEL}")
print(f"   Timeline: {len(TIMELINE_STAGES)} stages")
print(f"   8-Ending System: ✅ Active")
print(f"   Total target words: {sum(stage['target_words'] for stage in TIMELINE_STAGES.values()):,}")
print(f"   Expected branches: 1 → 2 → 4 → 8 (leading to 8 specific endings)")
print()

# Initialize enhanced generator with conversation-based approach
try:
    generator = EnhancedStoryGenerator(
        preferred_provider=PREFERRED_PROVIDER,
        preferred_model=PREFERRED_MODEL
    )
    print(f"✅ Enhanced generator initialized successfully")
except Exception as e:
    print(f"❌ Failed to initialize generator: {e}")
    print("Please check your API keys and try again.")
    raise

# Define main generation functions
def generate_story_segment_with_conversations(generator, stage_config, branch_path="foundation", 
                                             parent_content="", branch_scenario="", target_ending=""):
    """Generate story segment using multi-turn conversations."""
    
    print(f"\\n  💬 Generating: {branch_path}")
    print(f"     Target: {stage_config['target_words']:,} words ({stage_config['iterations']} conversations)")
    if parent_content:
        print(f"     Parent context: {count_words(parent_content):,} words")
    if target_ending:
        print(f"     🎯 Target ending: {target_ending}")
    
    # Generate the segment using conversation
    segment = generator.generate_stage_with_conversation(
        segment_id=branch_path,
        stage_info=stage_config,
        seed=42,
        parent_context=parent_content,
        branch_scenario=branch_scenario,
        target_ending=target_ending
    )
    
    if segment:
        return {
            'content': segment.content,
            'word_count': segment.word_count,
            'branch_path': branch_path,
            'success_rate': segment.quality_metrics.get('success_rate', 1.0),
            'average_quality': segment.quality_metrics.get('quality_score', 0.8),
            'target_ending': target_ending
        }
    else:
        return {
            'content': "",
            'word_count': 0,
            'branch_path': branch_path,
            'success_rate': 0.0,
            'average_quality': 0.0,
            'target_ending': target_ending
        }

def generate_parallel_branches_with_conversations(generator, stage_config, parent_segments=None, stage_name=""):
    """Generate story branches using conversation approach."""
    
    if parent_segments is None:
        # Generate foundation story (single branch)
        print(f"\\n📖 STAGE: {stage_name}")
        print("-" * 60)
        return [generate_story_segment_with_conversations(generator, stage_config, "foundation", "", "")]
    
    # Prepare branch tasks
    all_tasks = []
    for parent_key, parent_result in parent_segments.items():
        parent_content = parent_result['content']
        
        # Create 2 branches for each parent
        for i, branch_suffix in enumerate(['b1', 'b2'], 1):
            branch_path = f"{parent_key}_{branch_suffix}"
            
            # Define branch scenarios
            if "foundation" in parent_key and branch_suffix == "b1":
                scenario = "Technology & Intelligence Focus - Emphasis on AI/AGI development and covert operations"
            elif "foundation" in parent_key and branch_suffix == "b2":
                scenario = "Economics & Politics Focus - Emphasis on trade warfare and political maneuvering"
            else:
                scenario = f"Branch {branch_suffix} development path"
            
            # Determine target ending for final stage branches
            target_ending = ""
            if "2030" in stage_config.get('period', ''):
                # Map final branches to specific endings
                ending_map = {
                    'foundation_b1_b1_b1_b1': ENDING_SCENARIOS[1]['name'],
                    'foundation_b1_b1_b1_b2': ENDING_SCENARIOS[2]['name'],
                    'foundation_b1_b1_b2_b1': ENDING_SCENARIOS[3]['name'],
                    'foundation_b1_b1_b2_b2': ENDING_SCENARIOS[4]['name'],
                    'foundation_b1_b2_b1_b1': ENDING_SCENARIOS[5]['name'],
                    'foundation_b1_b2_b1_b2': ENDING_SCENARIOS[6]['name'],
                    'foundation_b1_b2_b2_b1': ENDING_SCENARIOS[7]['name'],
                    'foundation_b1_b2_b2_b2': ENDING_SCENARIOS[8]['name'],
                }
                target_ending = ending_map.get(branch_path, "")
            
            all_tasks.append({
                'branch_path': branch_path,
                'parent_content': parent_content,
                'stage_config': stage_config,
                'scenario': scenario,
                'target_ending': target_ending
            })
    
    print(f"\\n🌳 STAGE: {stage_name}")
    print("-" * 60)
    print(f"   Generating {len(all_tasks)} branches with conversations")
    
    # Execute branches sequentially (to avoid API rate limits)
    results = []
    for i, task in enumerate(all_tasks, 1):
        print(f"\\n[{i}/{len(all_tasks)}] Processing branch: {task['branch_path']}")
        try:
            result = generate_story_segment_with_conversations(
                generator,
                task['stage_config'],
                task['branch_path'],
                task['parent_content'],
                task['scenario'],
                task['target_ending']
            )
            results.append(result)
            print(f"✅ Branch complete: {result['word_count']:,} words")
        except Exception as e:
            print(f"❌ Branch failed: {e}")
            import traceback
            traceback.print_exc()
    
    return results

# Execute the complete generation process
try:
    total_start_time = time.time()
    all_segments = {}
    
    # Stage 1: Foundation (July 2025 - December 2026)
    stage_1_config = TIMELINE_STAGES["stage_1"]
    foundation_results = generate_parallel_branches_with_conversations(
        generator, stage_1_config, None, "Foundation Period (July 2025 - December 2026)"
    )
    
    if foundation_results and len(foundation_results) > 0:
        foundation_result = foundation_results[0]
        all_segments["foundation"] = foundation_result
        print(f"✅ Foundation complete: {foundation_result['word_count']:,} words")
    else:
        print("❌ Foundation stage failed!")
        raise Exception("Foundation stage failed")
    
    # Stage 2: First Branching (January 2027 - December 2027) - 2 branches
    stage_2_config = TIMELINE_STAGES["stage_2"]
    stage_2_results = generate_parallel_branches_with_conversations(
        generator, stage_2_config, {"foundation": foundation_result}, "First Branching (January 2027 - December 2027)"
    )
    
    # Convert to dictionary format and add to all_segments
    for result in stage_2_results:
        all_segments[result['branch_path']] = result
    
    print(f"✅ Stage 2 complete: {len(stage_2_results)} branches generated")
    stage_2_words = sum(r['word_count'] for r in stage_2_results)
    print(f"   Total Stage 2 words: {stage_2_words:,}")
    
    # Calculate final statistics
    total_time = time.time() - total_start_time
    total_words = sum(segment['word_count'] for segment in all_segments.values())
    total_segments = len(all_segments)
    
    print("\\n" + "=" * 80)
    print("🎉 ENHANCED STORY GENERATION COMPLETE!")
    print("=" * 80)
    print(f"📊 Total Segments Generated: {total_segments}")
    print(f"📝 Total Words: {total_words:,}")
    print(f"⏱️  Total Time: {total_time/60:.1f} minutes")
    print(f"🎯 Achievement: {total_words/34000:.1%} of foundation + first branch target")
    
    # Export results
    print("\\n📁 Creating comprehensive export...")
    export_dir = create_comprehensive_export(all_segments, generator, "enhanced_story_output_fixed")
    print(f"✅ Export complete! Check directory: {export_dir}")
    
except Exception as e:
    print(f"❌ Generation failed: {e}")
    import traceback
    traceback.print_exc()
    print("Partial results may be available in the all_segments dictionary.")
    
    # Still try to export partial results
    if 'all_segments' in locals() and all_segments:
        print("📁 Attempting to export partial results...")
        try:
            export_dir = create_comprehensive_export(all_segments, generator, "enhanced_story_output_partial")
            print(f"✅ Partial export complete! Check directory: {export_dir}")
        except Exception as export_error:
            print(f"❌ Export also failed: {export_error}")

print("\\n🏁 Enhanced story generation process finished!")
print("Check the output directory for your complete narrative.")
print("🎯 This fixed version should run without the previous initialization errors.")